# Formulas

A **formula** defines how a cost is calculated from energy data. Every formula has two key methods:

- **`get_values(start, end, resolution)`** — returns a time-indexed price series (€/MWh), for formulas whose price can be computed without meter readings.
- **`apply(meter, start, end, resolution)`** — multiplies the formula's price against actual meter readings and returns a cost series (€).

The library ships with the following formula types:

| Type | Description | `get_values` |
|---|---|:---:|
| `IndexFormula` | Price tracks one or more market indexes, plus an optional fixed component | ✓ |
| `ScheduledFormulas` | Different price per time window (time-of-use) | ✓ |
| `PeriodicFormula` | Fixed cost charged once per period (e.g. daily standing charge) | — |
| `TieredFormula` | Price depends on the consumption level within a period (bands) | — |
| `MinimumFormula` | Picks whichever sub-formula yields the lowest total cost for a period | — |
| `MaximumFormula` | Picks whichever sub-formula yields the highest total cost for a period | — |
| `MeterTypeFormula` | Routes to a different sub-formula based on the meter type | — |

## Setup — sample meter data

All examples share the same week of 15-minute meter readings. The `spot` price index is pre-registered so examples can reference it by name. A `show()` helper renders inline YAML as a highlighted code block.

In [1]:
import datetime as dt

import pandas as pd
import yaml
from pydantic import TypeAdapter

from energy_cost.formula import Formula
from energy_cost.index import CSVIndex, Index
from energy_cost.meter import Meter, MeterType, TimeseriesFrame

start = dt.datetime(2025, 12, 1, tzinfo=dt.UTC)
end   = dt.datetime(2025, 12, 8, tzinfo=dt.UTC)
res   = dt.timedelta(minutes=15)

# ~0.25 kWh per quarter-hour = 1 kW constant load
timestamps = pd.date_range(start, end, freq="15min", inclusive="left")
meter = Meter(measurements=TimeseriesFrame({"timestamp": timestamps, "value": 0.25}))

# Register the spot price index used by later examples
Index.register("spot", CSVIndex("../examples/indexes/foo.csv"))

# TypeAdapter is needed because Formula is a discriminated union, not a single model
formula_adapter = TypeAdapter(Formula)
def parse(yaml_str: str):
    """Parse a YAML string into a Formula object."""
    return formula_adapter.validate_python(yaml.safe_load(yaml_str))

## 1. IndexFormula

The most common formula type. The price is composed of:
- a `constant_cost` (fixed €/MWh component), and
- any number of `variable_costs`, each of which scales a registered index by a `scalar`.

A plain `IndexFormula` with only `constant_cost` is effectively a flat rate.

In [2]:
yaml_str = """\
constant_cost: 120.0
"""
flat_rate = parse(yaml_str)

# Price series — independent of meter data
flat_rate.get_values(start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,120.0
1,2025-12-01 00:15:00+00:00,120.0
2,2025-12-01 00:30:00+00:00,120.0
3,2025-12-01 00:45:00+00:00,120.0
4,2025-12-01 01:00:00+00:00,120.0


In [3]:
# Cost series — price × meter readings
flat_rate.apply(meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,30.0
1,2025-12-01 00:15:00+00:00,30.0
2,2025-12-01 00:30:00+00:00,30.0
3,2025-12-01 00:45:00+00:00,30.0
4,2025-12-01 01:00:00+00:00,30.0


To add a dynamic component, reference a pre-registered index by name in `variable_costs`.

In [4]:
yaml_str = """\
constant_cost: 10.0
variable_costs:
  - index: spot
    scalar: 1.05
"""
dynamic = parse(yaml_str)

dynamic.get_values(start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,115.0
1,2025-12-01 00:15:00+00:00,115.0
2,2025-12-01 00:30:00+00:00,115.0
3,2025-12-01 00:45:00+00:00,115.0
4,2025-12-01 01:00:00+00:00,115.0


## 2. ScheduledFormulas

A time-of-use formula that maps time windows to sub-formulas. Each entry in `schedule` has an optional `when` clause (days and/or time range); the first matching entry wins. A final entry without `when` acts as the catch-all default.

In [5]:
yaml_str = """\
schedule:
  - when:
      - start: "07:00:00"
        end: "23:00:00"
    formula:
      constant_cost: 150.0
  - formula:
      constant_cost: 80.0
"""
tou = parse(yaml_str)

# Show rows around the 07:00 transition (slots 24-35 = 06:00–09:00)
tou.get_values(start, end, res).iloc[24:36]

,timestamp,value
24,2025-12-01 06:00:00+00:00,80.0
25,2025-12-01 06:15:00+00:00,80.0
26,2025-12-01 06:30:00+00:00,80.0
27,2025-12-01 06:45:00+00:00,80.0
28,2025-12-01 07:00:00+00:00,150.0
29,2025-12-01 07:15:00+00:00,150.0
30,2025-12-01 07:30:00+00:00,150.0
31,2025-12-01 07:45:00+00:00,150.0
32,2025-12-01 08:00:00+00:00,150.0
33,2025-12-01 08:15:00+00:00,150.0


In [6]:
tou.apply(meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,20.0
1,2025-12-01 00:15:00+00:00,20.0
2,2025-12-01 00:30:00+00:00,20.0
3,2025-12-01 00:45:00+00:00,20.0
4,2025-12-01 01:00:00+00:00,20.0


## 3. PeriodicFormula

A fixed charge applied once per `period` (expressed as a `timedelta` or ISO 8601 duration). Typical uses: daily standing charges, monthly subscription fees.

`PeriodicFormula` does not implement `get_values()` because the per-kWh equivalent depends on actual consumption — use `apply()` instead.

In [7]:
yaml_str = """\
period: P1D
constant_cost: 1.50
"""
daily_charge = parse(yaml_str)

daily_charge.apply(meter, start, end, res).head(12)

,timestamp,value
0,2025-12-01 00:00:00+00:00,0.015625
1,2025-12-01 00:15:00+00:00,0.015625
2,2025-12-01 00:30:00+00:00,0.015625
3,2025-12-01 00:45:00+00:00,0.015625
4,2025-12-01 01:00:00+00:00,0.015625
5,2025-12-01 01:15:00+00:00,0.015625
6,2025-12-01 01:30:00+00:00,0.015625
7,2025-12-01 01:45:00+00:00,0.015625
8,2025-12-01 02:00:00+00:00,0.015625
9,2025-12-01 02:15:00+00:00,0.015625


## 4. TieredFormula

Applies different pricing based on how much energy is consumed within a `band_period`. Each `band` covers consumption up to an `up_to` threshold (MWh); the final band (no `up_to`) covers anything above.

There are two tiering modes:
- **`progressive`** (default) — each unit is priced according to the band it falls into, like income-tax brackets.
- **`banded`** — total consumption in the period determines the rate applied to *every* unit (cliff-edge pricing).

In [8]:
yaml_str = """\
band_period: P7D
bands:
  - up_to: 1.0
    formula:
      constant_cost: 80.0
  - formula:
      constant_cost: 120.0
"""
tiered_progressive = parse(yaml_str)

tiered_progressive.apply(meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,29.940476
1,2025-12-01 00:15:00+00:00,29.940476
2,2025-12-01 00:30:00+00:00,29.940476
3,2025-12-01 00:45:00+00:00,29.940476
4,2025-12-01 01:00:00+00:00,29.940476


With `mode="banded"`, the *entire* week's consumption determines which rate applies to every kWh. Here the meter uses 42 kWh/week (168 slots × 0.25 kWh), which exceeds 1 MWh, so all consumption is billed at 120 €/MWh.

In [9]:
yaml_str = """\
band_period: P7D
mode: banded
bands:
  - up_to: 1.0
    formula:
      constant_cost: 80.0
  - formula:
      constant_cost: 120.0
"""
tiered_banded = parse(yaml_str)

tiered_banded.apply(meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,30.0
1,2025-12-01 00:15:00+00:00,30.0
2,2025-12-01 00:30:00+00:00,30.0
3,2025-12-01 00:45:00+00:00,30.0
4,2025-12-01 01:00:00+00:00,30.0


## 5. MinimumFormula / MaximumFormula

These formulas evaluate a list of sub-formulas over a `period` and apply the one that produces the lowest (or highest) **total cost** for that period. This is useful for tariffs with a guaranteed floor or ceiling.

A typical use of `MinimumFormula`: "you pay at least a fixed flat rate, or the dynamic market rate — whichever is cheaper for the month".

In [10]:
yaml_str = """\
period: P7D
minimum:
  - constant_cost: 100.0
  - constant_cost: 10.0
    variable_costs:
      - index: spot
        scalar: 1.05
"""
cheapest = parse(yaml_str)

cheapest.apply(meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,25.0
1,2025-12-01 00:15:00+00:00,25.0
2,2025-12-01 00:30:00+00:00,25.0
3,2025-12-01 00:45:00+00:00,25.0
4,2025-12-01 01:00:00+00:00,25.0


In [11]:
yaml_str = """\
period: P7D
maximum:
  - constant_cost: 100.0
  - constant_cost: 10.0
    variable_costs:
      - index: spot
        scalar: 1.05
"""
most_expensive = parse(yaml_str)

most_expensive.apply(meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,28.75
1,2025-12-01 00:15:00+00:00,28.75
2,2025-12-01 00:30:00+00:00,28.75
3,2025-12-01 00:45:00+00:00,28.75
4,2025-12-01 01:00:00+00:00,28.75


## 6. MeterTypeFormula

Routes to a different sub-formula based on the `type` attribute of the meter. Useful when a tariff has separate pricing for single-rate or night-only meter channels. A `"default"` key acts as a fallback for any unspecified meter type.

In [12]:
yaml_str = """\
by_meter_type:
  night_only:
    constant_cost: 180.0
  single_rate:
    constant_cost: 90.0
  default:
    constant_cost: 130.0
"""
by_type = parse(yaml_str)

# Demonstrate with a night_only meter
night_meter = Meter(
    type=MeterType.NIGHT_ONLY,
    measurements=TimeseriesFrame({"timestamp": timestamps, "value": 0.25}),
)
by_type.apply(night_meter, start, end, res).head()

,timestamp,value
0,2025-12-01 00:00:00+00:00,45.0
1,2025-12-01 00:15:00+00:00,45.0
2,2025-12-01 00:30:00+00:00,45.0
3,2025-12-01 00:45:00+00:00,45.0
4,2025-12-01 01:00:00+00:00,45.0
